In [ ]:
import pandas as pd
import numpy as np
import os
from matplotlib import pyplot as plt
import scipy.stats
from decoding_utils import apply_condition_filter
import vbn_utils
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [ ]:
from notebook_utils import (
    beh_mat_from_stim_table,
    mean_beh_mat_across_sessions,
    get_omission_response_rate,
    get_post_omission_response_rate,
    get_shared_hit_rate,
    get_private_hit_rate,
    get_shared_fa_rate,
    get_private_fa_rate,
    get_shared_nonchange_response_rate,
    get_private_nonchange_response_rate,
    skip_diag_masking,
    get_session_engaged_dprime,
    get_session_engaged_hit_count,
    get_experience_session_id_for_mouse,
    calculate_metric_for_selection,
    paired_image_mat_from_stim_table,
)

## Data loading

In [ ]:
#Paths to all of the useful supplemental tables and tensors
stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_sessions_table.csv"

In [ ]:
units = pd.read_csv(unit_table_file)
unnamedcols = [c for c in units.columns if 'Unnamed' in c]
units = units.drop(columns=unnamedcols)

stim_table = pd.read_csv(stim_table_file)
stim_table = stim_table.drop(columns='Unnamed: 0')

sessions_table = pd.read_csv(sessions_table_file)

In [ ]:
good_sessions = sessions_table[sessions_table['abnormal_histology'].isnull() & sessions_table['abnormal_activity'].isnull()]
good_behavior_sessions = good_sessions[(good_sessions['engaged_dprime']>=1)&(good_sessions['engaged_hitcount']>=50)]

## Hit rates for holdover images on Familiar and Novel days

In [ ]:
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['pdf.fonttype'] = 42

In [ ]:
import warnings
warnings.filterwarnings("ignore")

for training_trajectory in ['GGH', 'GHG', 'HHG',]:
    beh_mat_dict = {'Familiar':[], 'Novel':[]}
    image_set_dict = {}
    for experience_level in ['Familiar', 'Novel']:
        sessions_to_analyze = good_sessions[apply_condition_filter(good_sessions, experience_level, training_trajectory)]['ecephys_session_id'].unique()

        ims, counts, beh_mats = mean_beh_mat_across_sessions(stim_table, sessions_to_analyze)
        beh_mat_dict[experience_level] = beh_mats
        image_set_dict[experience_level] = ims[0]

        
    fig, axes = plt.subplots(1,3)
    fig.set_size_inches(10,5)
    fig.suptitle(f'{training_trajectory}: {len(sessions_to_analyze)} sessions')
    for experience_level, ax in zip(['Familiar', 'Novel'], axes[:2]):
        beh_mats = beh_mat_dict[experience_level]
        im = ax.imshow(np.nanmean(beh_mats, axis=0), clim=[0,1])
        ax.set_xlabel('Change Image')
        ax.set_ylabel('Pre-change Image')
        ax.set_xticks(np.arange(8))
        ax.set_xticklabels(image_set_dict[experience_level], rotation=90)
        ax.set_yticks(np.arange(8))
        ax.set_yticklabels(image_set_dict[experience_level])
        
    holdover_responses_83 = {'Familiar':[], 'Novel':[]}
    holdover_responses_111 = {'Familiar':[], 'Novel':[]}
    for experience_level in ['Familiar', 'Novel']: 
        beh_mats = beh_mat_dict[experience_level]
        beh_mats = np.array([skip_diag_masking(b) for b in beh_mats])
        holdover_responses_83[experience_level].append(np.nanmean(beh_mats[:, :, 6], axis = 0))
        holdover_responses_111[experience_level].append(np.nanmean(beh_mats[:, :, 7], axis = 0))

    axes[2].plot(holdover_responses_83['Familiar'], holdover_responses_83['Novel'], 'ko')
    axes[2].plot(holdover_responses_111['Familiar'], holdover_responses_111['Novel'], 'ko', markerfacecolor='w', markeredgewidth=2)
    axes[2].plot([0,1], [0,1], 'k--')
    axes[2].set_aspect('equal')
    axes[2].set_xlabel('Hit rate during familiar sessions')
    axes[2].set_ylabel('Hit rate during novel sessions')

    plt.tight_layout()
    fig.savefig(os.path.join("/Volumes/programs/mindscope/workgroups/np-behavior/VBN Manuscript", f'behavioral_response_matrix_{training_trajectory}_test.png'))

In [ ]:
mouse_session_counts = good_sessions.value_counts('mouse_id')
mice_with_fam_and_nov_sessions = mouse_session_counts[mouse_session_counts > 1].index.values

In [ ]:
sessions_to_analyze = good_sessions['ecephys_session_id'].unique()

session_beh_mats = {}
beh_mat_array = []
count_array = []
session_labels = []
experience_labels = []
image_set_labels = []
for _, session in good_sessions.iterrows():
    session_id = session['ecephys_session_id']
    mouse_id = session['mouse_id']
    experience = session['experience_level']

    ims, counts, beh_mat = beh_mat_from_stim_table(stim_table, session_id)
    image_set = 'G' if 'im036_r' in ims else 'H'
    
    beh_mat_array.append(beh_mat)
    count_array.append(counts)
    session_labels.append(session_id)
    experience_labels.append(experience)
    image_set_labels.append(image_set)


    session_beh_mats[session_id] = {'beh_mat': beh_mat, 'counts': counts, 'images': ims}
    

In [ ]:
beh_mat_array = np.array(beh_mat_array)
beh_mat_no_diag_array = np.array([skip_diag_masking(b) for b in beh_mat_array])

count_array = np.array(count_array)
count_no_diag_array = np.array([skip_diag_masking(b) for b in count_array])

good_behavior_session_filter = np.array([(get_session_engaged_dprime(stim_table, session_id)>1) & \
                                (get_session_engaged_hit_count(stim_table, session_id)>50) for session_id in session_labels])

experience_labels = np.array(experience_labels)
session_labels = np.array(session_labels)
image_set_labels = np.array(image_set_labels)


## Hit rates on holdover images for Novel days for novel/familiar pre-images

In [ ]:
im_83_hit_rates = {'Familiar':[], 'Novel':[], 'Familiar_from_holdover':[], 'Novel_from_holdover':[], }
im_111_hit_rates = {'Familiar':[], 'Novel':[], 'Familiar_from_holdover':[], 'Novel_from_holdover':[], }

for mouse in mice_with_fam_and_nov_sessions:

    for experience in ['Familiar', 'Novel']:

        session_id = get_experience_session_id_for_mouse(sessions_table, mouse, experience)

        beh_mat = session_beh_mats[session_id]['beh_mat']
        counts = session_beh_mats[session_id]['counts']
        ims = session_beh_mats[session_id]['images']

        beh_mat_no_diag = skip_diag_masking(beh_mat)
        counts_no_diag = skip_diag_masking(counts)


        for ind, imrates in zip([6,7], [im_83_hit_rates[experience], im_111_hit_rates[experience]]):

            ind_count = np.nansum(counts_no_diag[:, ind])
            if (ind_count > 10) & (get_session_engaged_dprime(stim_table, session_id)>1) & (get_session_engaged_hit_count(stim_table, session_id)>50):

                #print(ind_count, get_session_engaged_dprime(stim_table, session_id), get_session_engaged_hit_count(stim_table, session_id))
                imrates.append(np.nanmean(beh_mat_no_diag[:, ind]))
            
            else:
                imrates.append(np.nan)

In [ ]:
fig, ax = plt.subplots()
ax.plot(im_83_hit_rates['Familiar'], im_83_hit_rates['Novel'], 'ko')
ax.plot(im_111_hit_rates['Familiar'], im_111_hit_rates['Novel'], 'ro')
ax.set_aspect('equal')
ax.legend(['im083', 'im111'])
ax.plot([0,1], [0,1], 'k--')
ax.set_xlabel('Hit rate during Familiar session')
ax.set_ylabel('Hit rate during Novel session')



In [ ]:
shared_to_shared_hit_rate = beh_mat_no_diag_array[(good_behavior_session_filter) & (experience_labels=='Novel')][:, 6, 6:]
nonshared_to_shared_hit_rate = beh_mat_no_diag_array[(good_behavior_session_filter) & (experience_labels=='Novel')][:, :6, 6:]

In [ ]:
null_shared_to_shared = []
null_nonshared_to_shared = []
for iteration in range(100):
    # Shuffle rows of each beh matrix
    beh_mat_no_diag_array_shuff = np.array([np.random.permutation(b) for b in beh_mat_no_diag_array[(good_behavior_session_filter) & (experience_labels=='Novel')]])
    stos = np.nanmean(beh_mat_no_diag_array_shuff[:, 6, 6:], axis=1)
    ntos = np.nanmean(beh_mat_no_diag_array_shuff[:, :6, 6:], axis=(1,2))
    null_shared_to_shared.extend(np.nanmean(beh_mat_no_diag_array_shuff[:, 6, 6:], axis=1))
    null_nonshared_to_shared.extend(np.nanmean(beh_mat_no_diag_array_shuff[:, :6, 6:], axis=(1,2)))

null_shared_to_shared = np.array(null_shared_to_shared)
null_nonshared_to_shared = np.array(null_nonshared_to_shared)


In [ ]:
fig, ax = plt.subplots()
ax.plot(null_shared_to_shared, null_nonshared_to_shared, 'ko', alpha=0.002)
ax.plot(np.mean(shared_to_shared_hit_rate, axis=1), np.mean(nonshared_to_shared_hit_rate, axis=(1,2)), 'ro')

ax.errorbar(np.nanmean(null_shared_to_shared), np.nanmean(null_nonshared_to_shared), 
                        xerr=np.nanstd(null_shared_to_shared), yerr=np.nanstd(null_nonshared_to_shared), color='k')
ax.errorbar(np.nanmean(shared_to_shared_hit_rate), np.nanmean(nonshared_to_shared_hit_rate), 
                        xerr=np.nanstd(shared_to_shared_hit_rate), yerr=np.nanstd(nonshared_to_shared_hit_rate), color='r')

ax.set_aspect('equal')
ax.plot([0,1], [0,1], 'k--')
ax.set_xlabel('From shared image')
ax.set_ylabel('From non-shared image')

scipy.stats.ranksums(null_shared_to_shared - null_nonshared_to_shared, np.mean(shared_to_shared_hit_rate, axis=1) - np.mean(nonshared_to_shared_hit_rate, axis=(1,2)), nan_policy='omit')


## Hit rates for non-omissions, omissions and post-omissions

A few notes about the stim_table columns:
- a lick bout is defined as any lick that follows the last lick by > 0.5 seconds
- `lick_time` is really 'lick bout time'. Not all licks are registered in this column, just the first licks in lick bouts. If you want to see all the licks in a trial, `lick_times` stores them in a (string) list
- `lick_for_flash` indicates whether a lick bout initiated after the start time of a flash and before the next flash. In the next cell we will add `lick_for_flash_during_response_window`, which will indicate whether a lick bout started in the response window after a flash (100-750 ms after flash start)
- `first_lick_in_trial` should be named 'first lick bout in trial'. There are some edge cases when consummatory licks from the last trial continue on into the next trial, but this trial doesn't abort immediately. In these cases, you can have a lick bout that starts after these consummatory licks, and `first_lick_in_trial` will be True, but `before_first_trial_lick` will be false (since consummatory licks happened earlier in the trial).



In [ ]:
response_rate_summary = {s:{col: np.nan for col in ['G_Familiar_private_nonchange', 'G_Familiar_shared_nonchange', 
                                                    'G_Novel_private_nonchange', 'G_Novel_shared_nonchange',
                                                    'H_Familiar_private_nonchange', 'H_Familiar_shared_nonchange', 
                                                    'H_Novel_private_nonchange', 'H_Novel_shared_nonchange',
                                                    'G_Familiar_private_hit', 'G_Familiar_shared_hit', 
                                                    'G_Novel_private_hit', 'G_Novel_shared_hit',
                                                    'H_Familiar_private_hit', 'H_Familiar_shared_hit', 
                                                    'H_Novel_private_hit', 'H_Novel_shared_hit',
                                                    'G_Familiar_private_fa', 'G_Familiar_shared_fa', 
                                                    'G_Novel_private_fa', 'G_Novel_shared_fa',
                                                    'H_Familiar_private_fa', 'H_Familiar_shared_fa', 
                                                    'H_Novel_private_fa', 'H_Novel_shared_fa',
                                                    'G_Familiar_omission', 'G_Novel_omission', 
                                                    'H_Familiar_omission', 'H_Novel_omission', 
                                                    'G_Familiar_postomission', 'G_Novel_postomission', 
                                                    'H_Familiar_postomission', 'H_Novel_postomission']} for s in good_behavior_sessions['ecephys_session_id'].values}

for isess, session in good_behavior_sessions.iterrows():
    
    experience_level = session['experience_level']
    image_set = session['image_set']
    session_id = session['ecephys_session_id']
    session_stim_table = stim_table[stim_table['session_id']==session_id]
    session_trials = session_stim_table.groupby('behavior_trial_id').head(1)

    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'private_nonchange'] = get_private_nonchange_response_rate(session_stim_table)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'private_hit'] = get_private_hit_rate(session_trials)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'private_fa'] = get_private_fa_rate(session_trials)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'shared_nonchange'] = get_shared_nonchange_response_rate(session_stim_table)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'shared_hit'] = get_shared_hit_rate(session_trials)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'shared_fa'] = get_shared_fa_rate(session_trials)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'omission'] = get_omission_response_rate(session_stim_table)
    response_rate_summary[session_id][image_set + '_' + experience_level + '_' + 'postomission'] = get_post_omission_response_rate(session_stim_table)

In [ ]:
response_rate_df = pd.DataFrame.from_dict(response_rate_summary, orient='index')

In [ ]:
aggregated_labels = ['Familiar_shared', 'Familiar_private', 'Novel_shared', 'Novel_private', 'Novel', 'Familiar']
response_categories = ['nonchange', 'hit',]# 'fa', 'omission', 'postomission']

cols_to_aggregate = []
for resp in response_categories:
    for ag_label in aggregated_labels:

        cols = [c for c in response_rate_df.columns if ag_label + '_' + resp in c]
        if len(cols)>0:
            cols_to_aggregate.append(cols)

In [ ]:
labels = []
figure_labels = []
values = []
means = []
sems = []
for cols in cols_to_aggregate:
    figure_labels.append(cols[0][2:].replace('shared', 'session \n holdover').replace('private', '').replace('_', ' ').replace('hit', 'change'))

    labels.append(cols[0][2:])
    vals = response_rate_df[cols].to_numpy().flatten()
    vals = vals[~np.isnan(vals)]

    values.append(vals)
    means.append(np.mean(vals))
    sems.append(np.std(vals)/len(vals)**0.5)

In [ ]:
plt.rcParams['font.size'] = 16
fig, ax = plt.subplots()
for ivals, vals in enumerate(values):
    color = 'b' if 'Familiar' in labels[ivals] else 'r'
    ax.boxplot(vals, positions=[ivals+1], widths=0.5, showfliers=False, whis=(10, 90), notch=True, boxprops={'color': color, 'linewidth': 2}, medianprops={'color': color, 'linewidth': 2}, whiskerprops={'color': color, 'linewidth': 2}, capprops={'color': color, 'linewidth': 2})

ax.set_xticklabels(figure_labels, rotation=90)
vbn_utils.formatFigure(fig, ax, yLabel='Response rate')
fig.savefig("/Volumes/programs/mindscope/workgroups/np-behavior/VBN Manuscript/behavioral_response_rate_boxplot_test.png")


### Statistical comparisons of response rates

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(7,7)
vbn_utils.plot_comparison_matrix(*values, colorbar=True, ax=ax, cmap='PiYG', binarize=True)
ax.set_xticklabels(figure_labels, rotation=90)
ax.set_yticklabels(figure_labels,)